# 4. Build a Simple Model and Assess Performance

**Homework — Group 34**

This notebook trains the model on the columns chosen in the feature-selection notebook (sections cited as 3.x), assesses
it on the same time folds used throughout, and writes the predictions for `test.csv` in the `sample_submission.csv`
format.

### 4.0 Setup and hand-off

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TARGET = "repeat_purchase_90d"
DATA_DIR = Path("data")
SUBMISSION_FILE = Path("submission.csv")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # IterativeImputer early-stopping notice
pd.set_option("display.width", 200)

In [ ]:
sel = joblib.load(DATA_DIR / "selected_features.joblib")
X_train, y, X_test = sel["X_train"], sel["y"], sel["X_test"]
cross_v, C = sel["cross_v"], sel["instrument_C"]
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train: {X_train.shape} | test: {X_test.shape} | selected columns: {len(sel['selected'])} | folds: {len(cross_v)}")
print(f"repeat rate in train: {y.mean():.3f}")

### 4.1 Assessment strategy and metric

- **Strategy: five time-ordered folds with a 90-day gap** (built in 2.6). Test lies after train in time (EDA 1.8), so each
  validation fold lies after its training rows, and training rows whose 90-day target window reaches into the validation
  period are dropped. Random K-fold would let the model learn from the future and overstate its score;
- **Main metric: F1 on the repeat class**, the competition's metric. It balances precision (of the customers predicted to
  repeat, how many do) and recall (of the customers who repeat, how many are found), so it rewards finding repeat
  customers without simply predicting "repeat" for everyone;
- **Second metric: ROC-AUC**, which judges how well the predicted probabilities rank customers, independently of the
  threshold. It was the measure used to make the preprocessing and selection choices;
- **Baseline: predict "repeat" for every row.** With 56% repeats this already scores F1 ≈ 0.72 (EDA 1.2), so a model is
  only useful if it beats it.

### 4.2 The model and its out-of-fold predictions

**Logistic regression with C = 0.01**: the model every earlier choice was measured with, so preprocessing, features and
model fit together; `C` was chosen on these folds in 2.8. The preprocessing recipe of the hand-off is part of the
pipeline, so it is refitted on the training part of each fold. Each fold's validation probabilities are kept to choose
the threshold.

In [ ]:
def make_model():
    return Pipeline([("prep", clone(sel["preprocessor"])),
                     ("clf", LogisticRegression(C=C, max_iter=5000))])


oof = []   # (true labels, predicted probabilities) of each validation fold
for tr, va in cross_v:
    model = make_model().fit(X_train.iloc[tr], y.iloc[tr])
    oof.append((y.iloc[va].to_numpy(), model.predict_proba(X_train.iloc[va])[:, 1]))

print("ROC-AUC per fold:", [round(roc_auc_score(y_va, p), 4) for y_va, p in oof])

### 4.3 Decision threshold

The model gives a probability; a customer is predicted to repeat when it is at or above the threshold. The default 0.5 is
not necessarily best for F1, so every threshold from 0.20 to 0.80 is applied to the out-of-fold probabilities and the one
with the highest mean F1 over the five folds is kept. Test is never used.

In [ ]:
THRESHOLDS = np.round(np.arange(0.20, 0.81, 0.01), 2)
f1_by_threshold = pd.DataFrame({t: [f1_score(y_va, (p >= t).astype(int)) for y_va, p in oof] for t in THRESHOLDS}).T
mean_f1 = f1_by_threshold.mean(axis=1)
BEST_T = float(mean_f1.idxmax())
baseline_f1 = np.mean([f1_score(y_va, np.ones_like(y_va)) for y_va, _ in oof])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mean_f1.index, mean_f1.values, color="#1779A8", label="logistic regression")
ax.axhline(baseline_f1, color="#C96A00", linestyle="--", label="baseline: always 'repeat'")
ax.axvline(BEST_T, color="grey", linestyle=":")
ax.set(xlabel="threshold", ylabel="mean F1 over the 5 folds", title="F1 against the decision threshold")
ax.legend()
plt.show()
print(f"best threshold: {BEST_T:.2f} | mean F1 {mean_f1[BEST_T]:.4f} (at 0.50: {mean_f1[0.5]:.4f}; baseline: {baseline_f1:.4f})")

<div class="alert alert-block alert-warning">

- The best threshold is **0.47**, with a mean F1 of 0.761, against 0.758 at the default 0.50: the curve is flat around 0.5, so the exact value matters little;
- Every threshold from 0.20 (the lowest tried) to 0.60 beats the baseline (0.721). The threshold is one number chosen on the same folds that report the score, so the F1 below is very slightly optimistic.

</div>

### 4.4 Performance per fold

The model at the chosen threshold against the baseline, fold by fold.

In [ ]:
rows = []
for k, (y_va, p) in enumerate(oof, 1):
    pred = (p >= BEST_T).astype(int)
    rows.append({"fold": k, "repeat rate": y_va.mean(), "AUC": roc_auc_score(y_va, p),
                 "F1 baseline": f1_score(y_va, np.ones_like(y_va)), "F1 at 0.50": f1_score(y_va, (p >= 0.5).astype(int)),
                 f"F1 at {BEST_T:.2f}": f1_score(y_va, pred),
                 "precision": precision_score(y_va, pred), "recall": recall_score(y_va, pred)})
results = pd.DataFrame(rows).set_index("fold")
results.loc["mean"] = results.mean()
print(results.round(4).to_string())

gain = results[f"F1 at {BEST_T:.2f}"].iloc[:-1] - results["F1 baseline"].iloc[:-1]
print(f"\nF1 gain over the baseline: {gain.mean():+.4f} ± {gain.std(ddof=1) / np.sqrt(len(gain)):.4f}, "
      f"ahead on {(gain > 0).sum()}/{len(gain)} folds")

<div class="alert alert-block alert-warning">

- **The model beats the baseline on all five folds**: mean F1 0.761 against 0.721 (+0.041 ± 0.004). ROC-AUC is between 0.78 and 0.80 on every fold, so the ranking quality is stable over time;
- Recall (0.81) is higher than precision (0.72): with F1 and 56% repeats, it pays to predict "repeat" a little more often than it happens;
- The folds with fewer repeats (2 and 4) have the lowest F1, for the model and the baseline alike: part of the score depends on how common repeats are in the period.

</div>

### 4.5 What the model relies on

The final model is fitted on all training rows. Numeric inputs are scaled, so their coefficients can be compared: a
positive coefficient raises the predicted chance of a repeat purchase.

In [ ]:
final_model = make_model().fit(X_train, y)
coef = pd.Series(final_model.named_steps["clf"].coef_[0],
                 index=final_model.named_steps["prep"].get_feature_names_out()).sort_values()
print("largest positive coefficients:\n", coef.tail(8)[::-1].round(3).to_string())
print("\nlargest negative coefficients:\n", coef.head(8).round(3).to_string())

<div class="alert alert-block alert-warning">

- **Purchase frequency drives the prediction**: purchase counts over 90 and 365 days and purchases per year have the largest positive coefficients, and a long mean gap between purchases the largest negative one among the numeric columns. This matches EDA 1.12-1.13;
- Current-order values matter less; line discounts and tax raise the chance of a repeat a little;
- Among categories, payment type 9 raises it and salesperson Master 4 lowers it, as seen in EDA 1.11.

</div>

### 4.6 Predictions for `test.csv`

The model fitted on all training rows predicts the test rows with the threshold chosen in 4.3. `ID` is kept unchanged and
the rows follow `sample_submission.csv`.

In [ ]:
proba_test = final_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({"ID": X_test.index, TARGET: (proba_test >= BEST_T).astype(int)})

assert list(submission.columns) == list(sample.columns), "columns must match sample_submission.csv"
assert submission["ID"].tolist() == sample["ID"].tolist(), "IDs must follow sample_submission.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

oof_share = np.mean(np.concatenate([p >= BEST_T for _, p in oof]))
print(f"saved {SUBMISSION_FILE}: {len(submission)} rows")
print(f"predicted 'repeat': {submission[TARGET].mean():.1%} of test rows (out-of-fold: {oof_share:.1%}; "
      f"actual repeat rate in train: {y.mean():.1%})")

<div class="alert alert-block alert-warning">

- `submission.csv` has 986 rows, the two columns of `sample_submission.csv` and the same `ID` order (checked by the `assert` lines);
- The model predicts "repeat" for 70% of test rows, more than in the folds (63%). Test comes right after the most recent training years, which already had more repeats than average (EDA 1.14), and 94.5% of test rows belong to customers seen in train (EDA 1.8), so a higher share is plausible.

</div>